In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import timedelta

# -----------------------------
# Configuration
# -----------------------------
fake = Faker("en_IN")
Faker.seed(42)
random.seed(42)
np.random.seed(42)

NUM_CUSTOMERS = 3000
NUM_DRIVERS = 600
NUM_TRIPS = 25000

cities = ["Delhi", "Noida", "Gurugram", "Ghaziabad"]

locations = [
    "Connaught Place",
    "Karol Bagh",
    "Rajouri Garden",
    "Janakpuri",
    "Dwarka",
    "Rohini",
    "Pitampura",
    "Model Town",
    "Civil Lines",
    "Kashmere Gate",
    "Chandni Chowk",
    "Red Fort",
    "Jama Masjid",
    "Daryaganj",
    "Lajpat Nagar",
    "South Extension",
    "Defence Colony",
    "Greater Kailash",
    "Nehru Place",
    "Kalkaji",
    "Saket",
    "Malviya Nagar",
    "Hauz Khas",
    "Green Park",
    "AIIMS",
    "INA",
    "Sarojini Nagar",
    "RK Puram",
    "Vasant Kunj",
    "Mahipalpur",
    "Airport T3",
    "IGI Airport T1",
    "Aerocity",
    "New Delhi Railway Station",
    "Old Delhi Railway Station",
    "Anand Vihar ISBT",
    "Anand Vihar Railway Station",
    "Mayur Vihar",
    "Laxmi Nagar",
    "Preet Vihar",
    "Shahdara",
    "Dilshad Garden",
    "Seelampur",
    "Yamuna Bank",
    "ITO",
    "Mandi House",
    "Pragati Maidan",
    "Akshardham",
    "Noida Sector 15",
    "Noida Sector 16",
    "Noida Sector 18",
    "Noida Sector 37",
    "Noida Sector 62",
    "Noida Sector 63",
    "Noida Sector 76",
    "Noida Sector 78",
    "Noida Sector 137",
    "Noida City Centre",
    "Botanical Garden",
    "Film City Noida",
    "Logix Mall",
    "DLF Mall of India",
    "Wave City Center",
    "Pari Chowk",
    "Knowledge Park",
    "Greater Noida West",
    "Alpha 1",
    "Beta 1",
    "Gamma 1",
    "Delta 1",
    "Ecotech",
    "Surajpur",
    "Bisrakh",
    "Crossings Republik",
    "Indirapuram",
    "Vaishali",
    "Kaushambi",
    "Vasundhara",
    "Raj Nagar Extension",
    "Ghaziabad Railway Station",
    "Mohan Nagar",
    "Sahibabad",
    "Loni",
    "Sector 14 Gurugram",
    "Sector 29 Gurugram",
    "Sector 45 Gurugram",
    "Sector 56 Gurugram",
    "Cyber City",
    "Cyber Hub",
    "MG Road",
    "IFFCO Chowk",
    "Huda City Centre",
    "Golf Course Road",
    "Sohna Road",
    "Udyog Vihar",
    "DLF Phase 1",
    "DLF Phase 2",
    "DLF Phase 3",
    "DLF Phase 4",
    "Manesar",
    "Ambience Mall"
]
vehicle_types = ["Mini","Sedan","SUV","Auto","Bike"]

payment_methods = [
    "UPI","upi","Upi",
    "Cash","cash","CASH",
    "Credit Card","Wallet"
]

statuses = ["Completed","Cancelled"]

cancel_reasons = [
    "Driver Cancelled",
    "Customer Cancelled",
    "No Driver Available"
]

# -----------------------------
# Customers
# -----------------------------

customers = []

for i in range(1, NUM_CUSTOMERS+1):

    customers.append([
        f"C{i:05d}",
        fake.name(),
        random.choice(["Male","Female"]),
        random.randint(18,65),
        random.choice(cities),
        fake.date_between("-4y","today")
    ])

customers = pd.DataFrame(customers,columns=[
    "customer_id","customer_name","gender",
    "age","city","signup_date"
])

# -----------------------------
# Drivers
# -----------------------------

drivers=[]

for i in range(1,NUM_DRIVERS+1):

    drivers.append([
        f"D{i:04d}",
        fake.name(),
        f"DL{random.randint(1,99):02d}{random.choice(['A','B','C'])}{random.randint(1000,9999)}",
        random.choice(vehicle_types),
        random.randint(1,15),
        random.choice(cities),
        fake.date_between("-8y","today"),
        round(random.uniform(3.5,5),1)
    ])

drivers=pd.DataFrame(drivers,columns=[
    "driver_id","driver_name","vehicle_number",
    "vehicle_type","experience_years",
    "city","joining_date","driver_rating"
])

# -----------------------------
# Trips
# -----------------------------

trip_rows=[]

start=pd.Timestamp("2025-01-01")

for i in range(1,NUM_TRIPS+1):

    cust=random.choice(customers.customer_id.tolist())
    drv=random.choice(drivers.driver_id.tolist())

    vehicle=drivers.loc[
        drivers.driver_id==drv,
        "vehicle_type"
    ].values[0]

    booking=start+timedelta(
        days=random.randint(0,364),
        hours=random.randint(0,23),
        minutes=random.randint(0,59)
    )

    pickup=random.choice(locations)

    drop=random.choice([x for x in locations if x!=pickup])

    distance=round(random.uniform(1,30),1)

    duration=int(distance*random.uniform(2.5,4.5))

    surge=round(random.choice([1,1,1,1.2,1.5,2.0]),1)

    fare=round((50+distance*15+duration*2)*surge,2)

    status=np.random.choice(
        statuses,
        p=[0.9,0.1]
    )

    payment=random.choice(payment_methods)

    if status=="Completed":

        cancel=""

        driver_rating=round(random.uniform(3,5),1)

        customer_rating=round(random.uniform(3,5),1)

    else:

        cancel=random.choice(cancel_reasons)

        driver_rating=np.nan

        customer_rating=np.nan

    trip_rows.append([
        f"T{i:06d}",
        cust,
        drv,
        booking,
        pickup,
        drop,
        distance,
        duration,
        vehicle,
        payment,
        surge,
        fare,
        status,
        cancel,
        driver_rating,
        customer_rating
    ])

trips=pd.DataFrame(trip_rows,columns=[
    "trip_id",
    "customer_id",
    "driver_id",
    "booking_datetime",
    "pickup_location",
    "drop_location",
    "distance_km",
    "duration_minutes",
    "vehicle_type",
    "payment_method",
    "surge_multiplier",
    "fare",
    "trip_status",
    "cancellation_reason",
    "driver_rating",
    "customer_rating"
])

# -----------------------------
# Dirty Data
# -----------------------------

# Duplicate rows (2%)
duplicates=trips.sample(frac=0.02,random_state=42)
trips=pd.concat([trips,duplicates],ignore_index=True)

# NULL payment
idx=trips.sample(frac=0.01).index
trips.loc[idx,"payment_method"]=np.nan

# Negative fare
idx=trips.sample(frac=0.01).index
trips.loc[idx,"fare"]=-abs(trips.loc[idx,"fare"])

# Zero distance
idx=trips.sample(frac=0.01).index
trips.loc[idx,"distance_km"]=0

# NULL ratings
idx=trips.sample(frac=0.02).index
trips.loc[idx,"driver_rating"]=np.nan

idx=trips.sample(frac=0.02).index
trips.loc[idx,"customer_rating"]=np.nan

# Outlier fares
idx=trips.sample(frac=0.005).index
trips.loc[idx,"fare"]=trips.loc[idx,"fare"]*5

# -----------------------------
# Save Files
# -----------------------------

customers.to_csv("customers.csv",index=False)

drivers.to_csv("drivers.csv",index=False)

trips.to_csv("trips.csv",index=False)

print("Customers :",len(customers))
print("Drivers :",len(drivers))
print("Trips :",len(trips))

print("\nCSV files generated successfully.")

Customers : 3000
Drivers : 600
Trips : 25500

CSV files generated successfully.
